# 图像翻译 · 考核代码实践项目

H&E 描述组织形态，配对 HER2-IHC 图像显示不同的染色信息。这个项目使用真实配对图像训练颜色预测模型，并对照真实参考判断误差。数据检查、训练循环和结果导出已经提供；带 `...` 的位置由你补全。Copy & Edit 后保存自己的副本。代码、结果分析和项目海报共同作为考核材料，不提供标准答案。

In [ ]:
from pathlib import Path
import os, json, random, copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.set_num_threads(min(4, os.cpu_count() or 1))
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('project_outputs')
WORK.mkdir(exist_ok=True)
SIZE, BATCH, EPOCHS, LR = 128, 32, 5, 0.001  # 修改尺寸或轮数后，重新运行后续单元格
print('device:', DEVICE, 'output:', WORK)
def show(images, titles, path=None):
    fig, axes = plt.subplots(1,len(images),figsize=(4*len(images),4),squeeze=False)
    for ax, im, title in zip(axes[0],images,titles):
        if isinstance(im,torch.Tensor): im=im.detach().cpu().permute(1,2,0).numpy()
        ax.imshow(im); ax.set_title(title); ax.axis('off')
    fig.tight_layout()
    if path: fig.savefig(path,dpi=150,bbox_inches='tight')
    plt.show(); plt.close(fig)
def ready(*items):
    ok = all(item is not Ellipsis for item in items)
    if not ok: print('本部分仍有 ... 待填；补全后重新运行本格及后续代码。')
    return ok

RUN_TRAINING = False  # 补全待填处后改为 True

## 数据与使用范围

Input 已添加 BCI 配对图像，官方训练部分 3896 对、测试部分 977 对，原始图像为 1024×1024。配对文件同名，HE 为输入、IHC 为目标。沿用官方 train/test，训练部分固定留出 20% 作为图块级验证集；这份文件清单没有可核实的患者分组，不能称作患者独立验证。

来源：Sheng et al., BCI: Breast Cancer Immunohistochemical Image Generation through Pyramid Pix2pix, CVPR Workshops 2022；bupt-ai-cz/BCI。镜像页面标注的 MIT 与作者数据许可不同，本课程按作者 BCI_LICENSE 限定用于非商业教学和研究，并保留引用。

In [ ]:
INPUT=Path(os.environ.get('KYDW_BCI_ROOT','/kaggle/input'))
roots=[INPUT] if (INPUT/'HE/train').exists() else [p.parent for p in INPUT.rglob('HE') if (p/'train').is_dir() and (p.parent/'IHC/train').is_dir()]
assert roots,'在 Add Input 中添加 breast-cancer-immunohistochemistry-bci-dataset。'
ROOT=roots[0]
def pairs(split):
    source={p.name:p for p in (ROOT/'HE'/split).glob('*.png')}
    target={p.name:p for p in (ROOT/'IHC'/split).glob('*.png')}
    assert source and source.keys()==target.keys(),'输入与目标的文件名未一一对应'
    return pd.DataFrame([dict(id=name,image=str(source[name]),target=str(target[name])) for name in sorted(source)])
train_all,test=pairs('train'),pairs('test')
train,val=train_test_split(train_all,test_size=.2,random_state=SEED)
train,val,test=[x.reset_index(drop=True) for x in [train,val,test]]
manifest=pd.concat([train.assign(split='train'),val.assign(split='val'),test.assign(split='test')])
for row in manifest.itertuples():
    with Image.open(row.image) as a,Image.open(row.target) as b: assert a.size==b.size,(row.id,a.size,b.size)
manifest.to_csv(WORK/'pairs.csv',index=False)
print('train / val / test:',len(train),len(val),len(test),'paired files:',len(manifest))

## 真实配对图与同位置裁剪

相同文件名只说明文件配成一对，组织结构是否对应还需要观察。改变 `SAMPLE_INDEX` 与 `BOX`，比较同一处组织边缘、空隙和核密集区域。裁剪框用原图像素坐标表示。

In [ ]:
SAMPLE_INDEX=0
BOX=(256,256,512,512)  # 左、上、右、下；输入和目标共用同一框
row=train.iloc[SAMPLE_INDEX]
a=Image.open(row.image).convert('RGB');b=Image.open(row.target).convert('RGB')
show([a,b],['HE input: '+row.id,'IHC reference'])
show([a.crop(BOX),b.crop(BOX)],['Same HE region','Same IHC region'])
print('original:',a.size,'crop:',BOX)

## 填空 1 · 输入与参考的数值范围

输入与目标都是 RGB 图片，模型输出采用 0—1 范围。填写将原始 8 位颜色转换到这一范围的除数。翻转的随机决定只生成一次，配对两侧共同使用。

In [ ]:
SCALE = ...  # 待填：8 位 RGB 转换为 0—1 的除数
class PairDataset(Dataset):
    def __init__(self,rows,flip=0):self.rows=rows;self.flip=flip
    def __len__(self):return len(self.rows)
    def __getitem__(self,i):
        row=self.rows.iloc[i]
        a=Image.open(row.image).convert('RGB').resize((SIZE,SIZE),Image.Resampling.BILINEAR)
        b=Image.open(row.target).convert('RGB').resize((SIZE,SIZE),Image.Resampling.BILINEAR)
        if random.random()<self.flip:
            a=a.transpose(Image.Transpose.FLIP_LEFT_RIGHT);b=b.transpose(Image.Transpose.FLIP_LEFT_RIGHT)
        arrays=[torch.from_numpy(np.asarray(im).copy()).permute(2,0,1).float()/SCALE for im in [a,b]]
        return *arrays,row.id
if ready(SCALE):
    x,y,_=PairDataset(train)[0];print('input / target:',x.shape,y.shape,'ranges:',x.min().item(),x.max().item(),y.min().item(),y.max().item());show([x,y],['Model input','Model target'])

## 填空 2 · 图像输出

网络保留空间位置，为每个像素预测目标图像的颜色分量。填写输出通道数；Sigmoid 将各输出值映射到 0—1。小型 U-Net 的编码、解码和跳跃连接已经提供。

网络中的 `BatchNorm2d` 在训练时按批次调整中间特征的数值尺度，预测时使用训练中保存的统计量；`train()` 与 `eval()` 分别控制两种状态。

In [ ]:
OUT_CHANNELS = ...  # 待填：目标图像的颜色通道数
class SmallUNet(nn.Module):
    def __init__(self,out_channels):
        super().__init__()
        def block(a,b): return nn.Sequential(nn.Conv2d(a,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU(),nn.Conv2d(b,b,3,padding=1),nn.BatchNorm2d(b),nn.ReLU())
        self.e1=block(3,8);self.e2=block(8,16);self.mid=block(16,32)
        self.d2=block(48,16);self.d1=block(24,8);self.out=nn.Conv2d(8,out_channels,1)
    def forward(self,x):
        a=self.e1(x);b=self.e2(F.max_pool2d(a,2));c=self.mid(F.max_pool2d(b,2))
        d=self.d2(torch.cat([F.interpolate(c,size=b.shape[-2:],mode='bilinear',align_corners=False),b],1))
        e=self.d1(torch.cat([F.interpolate(d,size=a.shape[-2:],mode='bilinear',align_corners=False),a],1))
        return self.out(e)

def make_model(): return nn.Sequential(SmallUNet(OUT_CHANNELS),nn.Sigmoid()).to(DEVICE)
if ready(OUT_CHANNELS):print(make_model()(torch.zeros(2,3,SIZE,SIZE,device=DEVICE)).shape)

## 填空 3 · 重建损失

损失逐位置比较预测与配对参考的颜色，采用绝对差的平均值。填写 `torch.nn.functional` 中对应的损失函数，保留函数本身，不加括号。

In [ ]:
LOSS_FN = ...  # 待填：平均绝对误差对应的 PyTorch 函数

## 完整训练循环

训练、验证和最佳权重保存已经提供。A 不翻转，B 同步水平翻转输入与目标；两种方案使用相同划分和训练预算。补全后启用训练，权重只根据验证损失选择。

In [ ]:
def epoch(model,loader,optimizer=None):
    model.train(optimizer is not None);total,count=0.,0
    with torch.set_grad_enabled(optimizer is not None):
        for batch in loader:
            x,target=batch[0].to(DEVICE),batch[1].to(DEVICE)
            pred=model(x)
            loss=LOSS_FN(pred,target)
            if optimizer is not None:
                optimizer.zero_grad();loss.backward();optimizer.step()
            total+=loss.item()*len(x);count+=len(x)
    return total/count

def run_experiment(name,flip):
    random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
    model=make_model()
    optimizer=torch.optim.Adam(model.parameters(),lr=LR)
    train_loader=DataLoader(PairDataset(train,flip),batch_size=BATCH,shuffle=True,num_workers=0)
    val_loader=DataLoader(PairDataset(val),batch_size=BATCH,num_workers=0)
    history=[];best=float('inf');folder=WORK/name;folder.mkdir(exist_ok=True)
    for step in range(EPOCHS):
        a=epoch(model,train_loader,optimizer);b=epoch(model,val_loader)
        history.append(dict(epoch=step+1,train_loss=a,val_loss=b));print(name,history[-1])
        if b<best:
            best=b;torch.save(model.state_dict(),folder/'best.pt')
    pd.DataFrame(history).to_csv(folder/'history.csv',index=False)
    (folder/'config.json').write_text(json.dumps(dict(seed=SEED,size=SIZE,batch=BATCH,epochs=EPOCHS,lr=LR,flip=flip,train=len(train),val=len(val),test=len(test)),indent=2))
    model.load_state_dict(torch.load(folder/'best.pt',map_location=DEVICE,weights_only=True));model.eval()
    return model,folder
models={}
if RUN_TRAINING and ready(SCALE,OUT_CHANNELS,LOSS_FN):
    for name,flip in [('A',0.0),('B',0.5)]: models[name]=run_experiment(name,flip)
else: print('补全待填处后，将 RUN_TRAINING 改为 True，再运行本格。A 与 B 只改变水平翻转概率。')

## 填空 4 · 每张图的误差

误差数组的形状为 `[批量,通道,高,宽]`。每张图需要得到一个 MAE，填写求平均的轴元组，保留批量轴。恒等参照直接把 H&E 输入作为输出，与训练模型在同一测试图块上比较。

In [ ]:
IMAGE_AXES = ...  # 待填：对单张图的通道和空间位置求平均所用的轴
if ready(IMAGE_AXES):
    assert isinstance(IMAGE_AXES,tuple)
    for name,(model,folder) in models.items():
        rows=[]
        with torch.no_grad():
            for x,y,ids in DataLoader(PairDataset(test),batch_size=BATCH):
                pred=model(x.to(DEVICE)).cpu()
                error=(pred-y).abs();mae=error.mean(dim=IMAGE_AXES)
                baseline=(x-y).abs().mean(dim=IMAGE_AXES)
                assert mae.shape==(len(x),),'每张图应保留一个数，请检查汇总轴。'
                rows.extend(dict(id=i,mae=float(m),identity_mae=float(b)) for i,m,b in zip(ids,mae,baseline))
        table=pd.DataFrame(rows);table.to_csv(folder/'metrics.csv',index=False)
        print(name,'test n:',len(table),table[['mae','identity_mae']].mean().to_dict())
        hist=pd.read_csv(folder/'history.csv');hist.plot(x='epoch',y=['train_loss','val_loss']);plt.savefig(folder/'curves.png',dpi=150);plt.show();plt.close()
        # 固定样例与误差较大的样例分别保存，便于解释选图依据
        selected=[(i,'fixed') for i in range(3)]
        selected += [(int(test.index[test.id==identifier][0]),'large-error') for identifier in table.nlargest(2,'mae').id]
        for i,kind in selected:
            x,y,identifier=PairDataset(test)[i]
            with torch.no_grad():pred=model(x[None].to(DEVICE))[0].cpu()
            heat=(pred-y).abs().mean(0).numpy()
            fig,ax=plt.subplots(1,4,figsize=(14,4))
            for a,im,title in zip(ax[:3],[x,y,pred],['HE','True IHC','Predicted IHC']):a.imshow(im.permute(1,2,0));a.set_title(title);a.axis('off')
            artist=ax[3].imshow(heat,cmap='magma',vmin=0,vmax=1);ax[3].set_title('RGB absolute error');ax[3].axis('off');fig.colorbar(artist,ax=ax[3],fraction=.046)
            fig.suptitle(f'{kind}: {identifier}');fig.savefig(folder/f'{kind}-{identifier}.png',dpi=150,bbox_inches='tight');plt.show();plt.close(fig)
    if len(models)==2:
        a=pd.read_csv(WORK/'A/metrics.csv');b=pd.read_csv(WORK/'B/metrics.csv');a.merge(b,on='id',suffixes=('_A','_B'),validate='one_to_one').to_csv(WORK/'comparison.csv',index=False)

## 项目海报

制作一张 A1 横向海报，介绍领域背景、方法、项目流程和自己的实际结果。中心图展示真实输入、真实参考、预测及同位置比较；文字说明数据划分、训练条件和可见误差。图像数值误差反映颜色差异，不能直接代表 HER2 临床分级准确率。

在 PowerPoint 中设置 84.1×59.4 厘米画布，四周边距约 2 厘米。统一中文微软雅黑、英文 Arial；标题 58—72 磅、分区标题 30—36 磅、正文与图注 22—26 磅。使用参考线、对齐与分布，图片保持比例。引用的事实与图片标明文献，个人结果保留样本编号。

保存 `学习海报.pptx`、`学习海报.pdf`、完成后的 Notebook 和主要输出。代码与海报均不提供标准答案。